In [1]:
# Cell 1 - pip installs
"""
!pip install transformers>=4.50.0
!pip install python-dotenv
!pip install accelerate
!pip install bitsandbytes
!pip install Pillow
"""

'\n!pip install transformers>=4.50.0\n!pip install python-dotenv\n!pip install accelerate\n!pip install bitsandbytes\n!pip install Pillow\n'

In [2]:
# Cell 2 - imports
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig
from huggingface_hub import login
import torch
import os
from dotenv import load_dotenv
from PIL import Image
import requests

# Cell 3 - Login and setup
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
login(hf_token)

model_name = "google/gemma-3-27b-it"

In [3]:
# Cell 4 - Load model with better quantization settings
print(f"🖥️ Detected {torch.cuda.device_count()} GPU(s)")

# הגדרת quantization משופרת
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # שונה מfloat16
    bnb_4bit_quant_type="nf4"
)

# טעינת המודל
print("🔄 Loading model...")
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # שונה מfloat16
    trust_remote_code=True,
    attn_implementation="eager"  # יותר יציב
).eval()

# טעינת הprocessor
processor = AutoProcessor.from_pretrained(model_name)

print("✅ Model and processor loaded successfully.")

🖥️ Detected 4 GPU(s)
🔄 Loading model...


Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Model and processor loaded successfully.


In [4]:
# Cell 5 - Manual inference function (EXACTLY AS BEFORE)
def generate_text(messages, max_new_tokens=100):
    """
    פונקציה לייצור טקסט עם טיפול טוב יותר בשגיאות
    """
    try:
        # עיבוד ההודעות
        inputs = processor.apply_chat_template(
            messages, 
            add_generation_prompt=True, 
            tokenize=True,
            return_dict=True, 
            return_tensors="pt"
        )
        
        # העברה למכשיר הנכון
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # בדיקת הinput
        print(f"Input shape: {inputs['input_ids'].shape}")
        print(f"Input device: {inputs['input_ids'].device}")
        
        input_len = inputs["input_ids"].shape[-1]
        
        # יצירת תגובה עם הגדרות בטוחות
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
                repetition_penalty=1.1,
                pad_token_id=processor.tokenizer.eos_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                use_cache=True
            )
        
        # פענוח רק החלק החדש
        generated_ids = outputs[0][input_len:]
        decoded = processor.tokenizer.decode(generated_ids, skip_special_tokens=True)
        
        return decoded.strip()
        
    except Exception as e:
        print(f"Error in generation: {e}")
        import traceback
        traceback.print_exc()
        return None

In [12]:
# Cell 6 - Test real text cleaning with few-shot examples
print("\n🧪 Testing real text cleaning with few-shot examples...")

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "אתה עוזר לניקוי טקסטים עבריים. קבל טקסטים עבריים רועשים שעשויים להכיל פגמי קידוד (&quot;), קטעי HTML, טלפון/אימייל, אימוג'ים, פרסומות, או תבניות. החזר רק את הטקסט המנוקה, תוך שמירה על המשמעות, בעברית, ללא הסברים."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": """נקה טקסטים עבריים מפגמי קידוד, תבניות HTML, פרסומות, מידע מיותר ותגיות.

דוגמה 1:
קלט: דיווח: תא דאעש שנחשף בירדן תכנן לפגוע באנשי עסקים ישראליים
© סופק על ידי מעריב תא דאעש... ____________________________________________________________ סרטונים שווים ב-MSN (BuzzVideos)
פלט: דיווח: תא דאעש שנחשף בירדן תכנן לפגוע באנשי עסקים ישראליים
תא דאעש שנחשף בנובמבר האחרון בירדן, תכנן בין היתר לפגוע באנשי עסקים ישראלים ברבת עמון...

דוגמה 2:
קלט: סוחר שהפיץ נפצים באשדוד וערים אחרות הופלל בוואטסאפ
אלה רוזנבלט... היי, בלוח החדש של אשדוד נט כבר ביקרת? כל הדירות למכירה/השכרה באשדוד... אולי יעניין אותך גם
פלט: סוחר שהפיץ נפצים באשדוד וערים אחרות הופלל בוואטסאפ
אלה רוזנבלט
מחירון לנפצים שהופץ באפליקציה ע"י צעיר ירושלמי הביא לתפיסתו בעת ביצוע העסקה...

דוגמה 3:
קלט: כל הז'יטונים על הברך של ויטור: משמעות התיקו של הפועל באר שבע עם בית""ר ירושלים... Follow @josifoon... <email> ליגת העל 2017/18 קבוצהמשנצתהפשעריםנק1מכבי תל אביב24156320-42512... <phone>בני יהודה...
פלט: כל הז'יטונים על הברך של ויטור: משמעות התיקו של הפועל באר שבע עם בית"ר ירושלים
בינואר הימרה באר שבע על הכשירות של הפורטוגלי ולא חיזקה את ההגנה...

דוגמה 4:
קלט: הטבות המס שנועדו להחזיר ישראלים מעודדים אותם לרדת... ביקורת קשה נמתחה היום בוועדה לביקורת המדינה כנגד הטבות המס המפליגות ופטור מדיווח הניתנים לתושבים חוזרים ועולים חדשים...
פלט: הטבות המס שנועדו להחזיר ישראלים מעודדים אותם לרדת
ביקורת קשה נמתחה היום בוועדה לביקורת המדינה כנגד הטבות המס המפליגות ופטור מדיווח הניתנים לתושבים חוזרים ועולים חדשים...

עכשיו נקה את הטקסט הבא:
דרעי: אין סיבה שניכנס לעימותים בקואליציה סביב חוק הגיוס
יו"ר סיעת ש"ס ח"כ אריה דרעי אמר היום (שני) כי ביום רביעי יובא חוק הגיוס לקריאה טרומית, "אין כאן שינוי ומקווה שמרכיבי הקואליציה יבינו שכמו שהיינו שותפים הכי נאמנים ויציבים בקואליציה כך נמשיך ולא תהיה סיבה שנכנס לעימותים בסיפור הזה". תגיות: אריה דרעי חוק הגיוס

השב רק עם הטקסט המנוקה:"""}
        ]
    }
]

result = generate_text(messages, max_new_tokens=200)
print(result)


🧪 Testing real text cleaning with few-shot examples...
Input shape: torch.Size([1, 985])
Input device: cuda:0
דרעי: אין סיבה שניכנס לעימותים בקואליציה סביב חוק הגיוס
יו"ר סיעת ש"ס ח"כ אריה דרעי אמר היום (שני) כי ביום רביעי יובא חוק הגיוס לקריאה טרומית, "אין כאן שינוי ומקווה שמרכיבי הקואליציה יבינו שכמו שהיינו שותפים הכי נאמנים ויציבים בקואליציה כך נמשיך ולא תהיה סיבה שנכנס לעימותים בסיפור הזה".


In [6]:
# Cell 7 - Debug tokenizer (EXACTLY AS BEFORE)
def debug_tokenizer():
    print("\n🔍 Debug tokenizer...")
    
    # בדיקת הtokenizer
    print(f"Vocabulary size: {len(processor.tokenizer)}")
    print(f"EOS token: {processor.tokenizer.eos_token} (ID: {processor.tokenizer.eos_token_id})")
    print(f"Pad token: {processor.tokenizer.pad_token}")
    
    # בדיקת tokenization פשוטה
    test_text = "Hello world"
    tokens = processor.tokenizer.encode(test_text)
    decoded = processor.tokenizer.decode(tokens)
    print(f"Test encode/decode: '{test_text}' -> {tokens} -> '{decoded}'")

debug_tokenizer()


🔍 Debug tokenizer...
Vocabulary size: 262145
EOS token: <eos> (ID: 1)
Pad token: <pad>
Test encode/decode: 'Hello world' -> [2, 9259, 1902] -> '<bos>Hello world'


In [7]:
# Cell 8 - Memory monitoring (EXACTLY AS BEFORE)
def monitor_gpu_memory():
    print("\n🖥️ GPU Memory Status:")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
        print(f"  Memory reserved: {torch.cuda.memory_reserved(i) / 1024**3:.2f} GB")
        print("  ---")

monitor_gpu_memory()


🖥️ GPU Memory Status:
GPU 0: NVIDIA A10G
  Memory allocated: 3.27 GB
  Memory reserved: 3.87 GB
  ---
GPU 1: NVIDIA A10G
  Memory allocated: 3.73 GB
  Memory reserved: 4.52 GB
  ---
GPU 2: NVIDIA A10G
  Memory allocated: 3.73 GB
  Memory reserved: 4.52 GB
  ---
GPU 3: NVIDIA A10G
  Memory allocated: 4.98 GB
  Memory reserved: 6.08 GB
  ---
